In [4]:
import sys, platform, subprocess
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Git commit:", subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip())

Python: 3.7.17
Platform: Linux-7.0.0-31-generic-x86_64-with-debian-trixie-sid
Git commit: c76b4e2d5a43a1abe558610e39561b68e4fca5b0


In [5]:
import pandas as pd

DATA_PATH = Path("..") / "assets" / "data" / "Fish.csv"
df = pd.read_csv(DATA_PATH)

print("shape:", df.shape)
print("columns:", list(df.columns))
df.head()

shape: (159, 7)
columns: ['Species', 'Weight', 'Length1', 'Length2', 'Length3', 'Height', 'Width']


,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340


In [6]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(dropna=False),
    "min": df.select_dtypes("number").min(),
    "max": df.select_dtypes("number").max(),
})
quality

,dtype,missing,unique,min,max
Height,float64,0,154,1.7284,18.957
Length1,float64,0,116,7.5000,59.000
Length2,float64,0,93,8.4000,63.400
Length3,float64,0,124,8.8000,68.000
Species,object,0,7,NaN,NaN
Weight,float64,0,101,0.0000,1650.000
Width,float64,0,152,1.0476,8.142


In [7]:
df["Species"].value_counts()

Perch        56
Bream        35
Roach        20
Pike         17
Smelt        14
Parkki       11
Whitefish     6
Name: Species, dtype: int64

In [8]:
# --- Автоматические проверки качества (шаг 3.3) ---
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))

import importlib
import data_checks as dc
importlib.reload(dc)

report = dc.run_all_checks(df)

In [9]:
print("ФОРМА:")
print(report["shape"])

print("\nТИПЫ:")
print(report["dtypes"])

print("\nДУБЛИКАТЫ:")
print(report["duplicates"])

print("\nПРОПУСКИ:")
print(report["missing"])

print("\nБЕСКОНЕЧНОСТИ:")
print(report["infinities"])

print("\nНЕВОЗМОЖНЫЕ ЗНАЧЕНИЯ (<= 0):")
for col, info in report["positivity"].items():
    print(f"  {col}: n_violations={info['n_violations']}, min={info['min_value']}, indices={info['indices']}")

print("\nСОГЛАСОВАННОСТЬ ДЛИН (Length1 <= Length2 <= Length3):")
print(report["length_order"])

print("\nСОГЛАСОВАННОСТЬ Height/Width:")
print(report["volume_consistency"])

print("\nРАСПРЕДЕЛЕНИЕ ПО ВИДАМ:")
print(report["species_dist"])

print("\nРЕДКИЕ КАТЕГОРИИ (<10%):")
print(report["rare_categories"])

print("\nДИСБАЛАНС ВИДОВ:")
print(report["imbalance"])

print("\nПОЧТИ ОДИНАКОВЫЕ СТРОКИ:")
print(report["near_duplicates"])

print("\nПОЛЯ-КАНДИДАТЫ В ID:")
print(report["id_like"])

ФОРМА:
{'check': 'shape', 'rows': 159, 'cols': 7, 'rows_ok': True, 'cols_ok': True}

ТИПЫ:
           dtype
Species   object
Weight   float64
Length1  float64
Length2  float64
Length3  float64
Height   float64
Width    float64

ДУБЛИКАТЫ:
{'check': 'duplicates', 'subset': None, 'n_duplicated_rows': 0, 'duplicated_indices': []}

ПРОПУСКИ:
         missing  missing_pct
Species        0          0.0
Weight         0          0.0
Length1        0          0.0
Length2        0          0.0
Length3        0          0.0
Height         0          0.0
Width          0          0.0

БЕСКОНЕЧНОСТИ:
         pos_inf  neg_inf
Weight         0        0
Length1        0        0
Length2        0        0
Length3        0        0
Height         0        0
Width          0        0

НЕВОЗМОЖНЫЕ ЗНАЧЕНИЯ (<= 0):
  Weight: n_violations=1, min=0.0, indices=[40]
  Length1: n_violations=0, min=7.5, indices=[]
  Length2: n_violations=0, min=8.4, indices=[]
  Length3: n_violations=0, min=8.8, indices=[]
  H